In [443]:
import pandas as pd
from itertools import permutations
import re

import os

# INCOME CLASSIFICATION

A row is uniquely determined by 'Province', 'Area', 'Level'.

In [444]:
income_sheets = ['By-LGU-ARI-and-Dependencies-2023.xlsx', 
 'By-LGU-ARI-and-Dependencies-2024.xlsx', 
 'By-LGU-ARI-and-Dependencies-2025Preliminary.xlsx'
]

filepath_income_sheets = [os.path.join('data/income', sheet)for sheet in income_sheets]

In [445]:
year = 2023
filepath = filepath_income_sheets[0]

def extract_income(filepath, year):
    cols_take = list(range(2, 5)) + [23]
    income_df = pd.read_excel(
        filepath,
        skiprows=9
    ).iloc[:, cols_take]

    income_df.columns = ['Province', 'Area', 'Level', 'Income']

    # Standardize the Area name for cities
    income_df.Area = income_df.Area.astype(str).apply(change_city_name)

    income_df['Year'] = year

    income_df['Area'] = income_df['Area'].apply(clean_area)
    income_df['Province'] = income_df['Province'].apply(clean_area)


    income_df['Formal'] = income_df.Province + ', ' + income_df.Area + ', ' + income_df.Level

    return income_df.dropna()

def clean_area(area):
    # convert to string, strip, uppercase
    area = str(area).strip().upper()

    # replace ñ / Ñ with n
    area = re.sub(r'[ñÑ]', 'N', area)

    # convert "X CITY" -> "CITY OF X"
    if area.endswith(' CITY'):
        name = area.replace(' CITY', '')
        area = f'CITY OF {name}'

    return area

In [446]:
incomes_df = [extract_income(filepath_income_sheets[i], 2023 + i) for i in range(3)]

# fix the spelling mistake
incomes_df[0]['Area'] = incomes_df[0]['Area'].replace({
    'San Policarpo': 'San Policarpio'
})

mean_income_df = (
    pd.concat(incomes_df)
    .groupby(['Province', 'Area', 'Level'], as_index=False)['Income']
    .mean()
    .rename(columns={'Income': 'Mean_Income'})
)

In [447]:
for income_df_pairs in list(permutations(incomes_df, r=2),):
    income_df1, income_df2 = income_df_pairs
    differences = set(income_df1.Formal.to_list()) - set(income_df1.Formal.to_list())
    if not differences:
        print('They agree!')
    else:
        print('They disagree...huhuhuhu')

They agree!
They agree!
They agree!
They agree!
They agree!
They agree!


In [448]:
def classify_income(row):
    inc = row['Mean_Income']
    level = row['Level']
    
    # PROVINCES
    if level == 'Province':
        if inc >= 1_500_000_000:
            return '1st'
        elif inc >= 900_000_000:
            return '2nd'
        elif inc >= 700_000_000:
            return '3rd'
        elif inc >= 500_000_000:
            return '4th'
        else:
            return '5th'
    
    # CITIES
    elif level == 'City':
        if inc >= 1_300_000_000:
            return '1st'
        elif inc >= 1_000_000_000:
            return '2nd'
        elif inc >= 800_000_000:
            return '3rd'
        elif inc >= 500_000_000:
            return '4th'
        else:
            return '5th'
    
    # MUNICIPALITIES
    elif level == 'Municipality':
        if inc >= 200_000_000:
            return '1st'
        elif inc >= 160_000_000:
            return '2nd'
        elif inc >= 130_000_000:
            return '3rd'
        elif inc >= 90_000_000:
            return '4th'
        else:
            return '5th'
    
    else:
        return None

In [485]:
mean_income_df['Income_Class'] = mean_income_df.apply(classify_income, axis=1)

# fix/standardize North Cotabato to Cotabato
mean_income_df.Province = mean_income_df.Province.replace({'NORTH COTABATO':'COTABATO'})

In [523]:
mean_income_df.Level.value_counts()

Level
Municipality    1486
City             149
Province          82
Name: count, dtype: int64

# PSGC

In [554]:
psgc_df = pd.read_excel('data/income/psgc-1q-2025-publication-datafile.xlsx', sheet_name='PSGC')
psgc_df['10-digit PSGC'] = psgc_df['10-digit PSGC'].astype(str).str.zfill(10)
psgc_df = psgc_df[['10-digit PSGC', 'Name', 'Geographic Level']]
psgc_df.columns = ['PSGC', 'Area', 'Level']
psgc_df = psgc_df[
    (psgc_df.Level == 'Mun') |
    (psgc_df.Level== 'City') | 
    (psgc_df.Level == 'Prov')
].copy()

psgc_df
psgc_df.Area = psgc_df.Area.apply(clean_area)

In [ ]:
prov_psgc_df = psgc_df[psgc_df.Level == 'Prov'][['PSGC', 'Area']]
prov_psgc_df['Prov_Code'] = prov_psgc_df.PSGC.str[:5]
prov_map = dict(zip(prov_psgc_df.Prov_Code, prov_psgc_df.Area))

psgc_df['Prov_Code'] = psgc_df.PSGC.str[:5]
psgc_df['Province'] = psgc_df.Prov_Code.map(prov_map)

is_huc = psgc_df['Province'].isnull()
psgc_df.loc['Level', is_huc] = 'HUC'
# force all NCR cities to have a province of metro manila
psgc_df.loc[
    psgc_df['PSGC'].str.startswith('13'),
    'Province'
] = 'METRO MANILA'

# and all other HUCs are to be just have it as which province they are geographically

2879              CITY OF BAGUIO
12007            CITY OF ANGELES
12041           CITY OF OLONGAPO
16166             CITY OF LUCENA
17672    CITY OF PUERTO PRINCESA
24646             CITY OF ILOILO
26185            CITY OF BACOLOD
28523               CITY OF CEBU
28604          CITY OF LAPU-LAPU
28635            CITY OF MANDAUE
33039           CITY OF TACLOBAN
35014            CITY OF ISABELA
35060          CITY OF ZAMBOANGA
37154     CITY OF CAGAYAN DE ORO
37235             CITY OF ILIGAN
38314              CITY OF DAVAO
39621     CITY OF GENERAL SANTOS
40952             CITY OF BUTUAN
43698                  KAPALAWAN
43706               OLD KAABAKAN
43714                 KADAYANGAN
43722                  NABALAWAG
43730                 PAHAMUDDIN
43743                  MALIDEGAO
43751                  LIGAWASAN
43759                    TUGUNAN
Name: Area, dtype: object

In [561]:
psgc_df.loc[
    psgc_df['PSGC'].str.startswith('13'),
    'Province'
] = 'METRO MANILA'